# RAG-based Document Assistant
### Candidate: Alwin Hemanth KS

**Project Overview:**
This project implements a Retrieval-Augmented Generation (RAG) system using **LangChain**, **FAISS**, and **Groq**. It allows users to ask questions about local text documents and receive accurate answers based on the retrieved context.

## 1. Environment Setup
We begin by verifying that the core libraries (LangChain, FAISS, and Sentence Transformers) are correctly installed in the local environment.

In [6]:
try:
    import langchain
    import faiss
    import sentence_transformers
    print("✅ Success! Your RAG system environment is ready.")
except ImportError as e:
    print(f"❌ Still missing: {e}")

✅ Success! Your RAG system environment is ready.


## 2. Document Loading & Chunking
In this step, we load text files from the local directory. To prepare the text for the LLM, we split the documents into smaller chunks.

**Design Choice - Chunking:**
- **Chunk Size:** 500 characters.
- **Chunk Overlap:** 50 characters.
- **Reasoning:** This size ensures that each chunk is small enough for efficient retrieval but large enough to maintain semantic meaning. The overlap prevents important context from being lost at the split points.

In [7]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use 'r' before the string for Windows paths
path = r"D:\Doc" 

# 1. Load your documents from your D: drive
loader = DirectoryLoader(path, glob="./*.txt", loader_cls=TextLoader)
documents = loader.load()

# 2. Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)

print(f"✅ Successfully loaded files from {path}")
print(f"✅ Split into {len(chunks)} chunks.")

✅ Successfully loaded files from D:\Doc
✅ Split into 6 chunks.


## 3. Vector Embeddings & Storage
We transform the text chunks into mathematical vectors using a pre-trained embedding model and store them in a FAISS index for high-speed similarity searching.

**Design Choice - Embeddings & Vector Store:**
- **Model:** `sentence-transformers/all-MiniLM-L6-v2`. This model is chosen for its efficiency and strong performance on CPU-based systems.
- **Vector Store:** **FAISS** (Facebook AI Similarity Search). It is used because of its industry-standard performance in similarity search.
- **Similarity Metric:** Euclidean Distance (L2), which is the standard for mapping closeness in high-dimensional vector spaces.

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Initialize the Embedding Model
# We use 'all-MiniLM-L6-v2' because it's fast, free, and runs on your CPU
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Create the FAISS Vector Store (your searchable database)
# This will take your 6 chunks and turn them into vectors
vector_db = FAISS.from_documents(chunks, embeddings)

print("✅ Success! Your 6 chunks are now embedded and stored in FAISS.")

✅ Success! Your 6 chunks are now embedded and stored in FAISS.


## 4. RAG Pipeline Implementation
We use the **LangChain Expression Language (LCEL)** to build a pipeline that:
1. Retrieves the most relevant chunks from FAISS based on the user's question.
2. Passes those chunks as "context" to the Groq LLM.
3. Generates a final answer based **only** on the provided context.

**LLM Choice:**
- **Model:** `llama-3.1-8b-instant` via **Groq Cloud**. This model provides near-instant inference speeds, making it ideal for real-time RAG applications.

In [9]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Setup
# SECURITY NOTE: The Groq API Key has been removed for public GitHub submission.
# To run this notebook, please replace the placeholder below with your actual 
# 'gsk_...' key from https://console.groq.com/

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

# 2. Define a simple prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Create the RAG Chain using the pipe (|) operator
# This is the modern standard for LangChain v1.x
rag_chain = (
    {"context": vector_db.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Test it
query = "What is the main summary of the documents?"
response = rag_chain.invoke(query)
print(f"Answer: {response}")

Answer: The main summary of the documents is about the development and functionality of Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG) systems. 

RAG systems allow LLMs to access specific, up-to-date data without needing to retrain the model by providing relevant document "chunks" as context. 

LLMs, such as GPT-4, Claude, and Gemini, use the Transformer architecture and Self-Attention mechanism to process text and understand relationships between words, enabling them to generate answers based on facts rather than just their internal training data.


## 5. Demonstration
Below are two examples of the RAG system answering questions based on the retrieved document context.

In [12]:
# Question 2 Demonstration
query_2 = "How do RAG systems help LLMs avoid retraining?"
response_2 = rag_chain.invoke(query_2)
print(f"Question: {query_2}")
print(f"Answer: {response_2}")

Question: How do RAG systems help LLMs avoid retraining?
Answer: RAG systems help LLMs avoid retraining by giving them access to specific, up-to-date data without needing to retrain the model. This is achieved by first converting documents into numerical vectors using an embedding model, storing these vectors in a vector database, and then searching the database for the most relevant document "chunks" when a user asks a question. These chunks are then provided to the LLM as context, allowing it to generate an answer based on facts.
